In [0]:
from pyspark.sql.functions import col

# ============================================
# READ BRONZE DATA
# ============================================

bronze_orders_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/lakehouse_raw_data/bronze/orders"
)

# ============================================
# READ SILVER DATA
# ============================================

silver_orders_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/lakehouse_raw_data/silver/orders_clean"
)

# ============================================
# SOURCE COUNTS
# ============================================

bronze_count = bronze_orders_df.count()

silver_count = silver_orders_df.count()

# ============================================
# FAILED RECORDS
# ============================================

failed_records_df = bronze_orders_df.filter(
    (col("amount") <= 0)
    |
    (col("status") != "SUCCESS")
)

failed_count = failed_records_df.count()

# ============================================
# VALID RECORDS
# ============================================

valid_records_df = bronze_orders_df.filter(
    (col("amount") > 0)
    &
    (col("status") == "SUCCESS")
)

# ============================================
# DUPLICATE RECORD CHECK
# ONLY CHECK VALID RECORDS
# ============================================

duplicate_count = (
    valid_records_df.count()
    -
    valid_records_df.dropDuplicates(["order_id"]).count()
)

# ============================================
# RECONCILIATION SUMMARY
# ============================================

print("========== RECONCILIATION REPORT ==========")

print(f"Bronze Record Count: {bronze_count}")

print(f"Silver Record Count: {silver_count}")

print(f"Rejected Record Count: {failed_count}")

print(f"Duplicate Record Count: {duplicate_count}")

# ============================================
# DISPLAY FAILED RECORDS
# ============================================

print("Rejected Records")

display(failed_records_df)

# ============================================
# EXPECTED SILVER COUNT
# ============================================

expected_silver_count = (
    bronze_count
    -
    failed_count
    -
    duplicate_count
)

print(f"Expected Silver Count: {expected_silver_count}")

# ============================================
# FINAL RECONCILIATION STATUS
# ============================================

if silver_count == expected_silver_count:
    print("RECONCILIATION SUCCESSFUL")
else:
    print("RECONCILIATION FAILED")

========== RECONCILIATION REPORT ==========
Bronze Record Count: 100500
Silver Record Count: 100000
Rejected Record Count: 15054
Duplicate Record Count: 425
Rejected Records


amount,customer_id,order_id,order_time,status
-3554.03,5570.0,2,2025-11-29 17:15:00,FAILED
1965.02,237.0,23,2025-06-11 03:37:00,PENDING
4348.67,4877.0,29,2025-09-22 18:38:00,PENDING
-283.27,856.0,35,2025-06-20 03:14:00,FAILED
-4584.94,9333.0,40,2025-09-22 01:41:00,FAILED
-333.87,9739.0,41,2025-03-08 05:45:00,FAILED
-1231.17,1136.0,46,2025-11-26 02:04:00,FAILED
-4203.07,1486.0,52,2025-08-06 01:33:00,FAILED
2174.56,6082.0,53,2025-01-15 18:59:00,PENDING
3689.61,718.0,58,2025-07-15 05:57:00,PENDING


Expected Silver Count: 85021
RECONCILIATION FAILED
